# This Notebook combines the whole Processing Pipeline.

Components:
- Experiment setup, controlled through ```EXPERIMENT_CONFIG```
- Data loading
- Object initialization
- Preprocessing: Flatfield correction and perspective correction
- Segmentation
- Postprocessing
- Sensitivity Analysis

## Usage:
Update ```EXPERIMENT_CONFIG``` as needed, then run.

In [ ]:
chip_region = 1

pretrained_checkpoint = "../../sam2/checkpoints/sam2.1_hiera_base_plus.pt"
finetuned_checkpoint = "../../models/SAM2-IoU/checkpoints/checkpoint_230.pt"
model_cfg_b_plus = "configs/sam2.1/sam2.1_hiera_b+.yaml"

EXPERIMENT_CONFIG = {
    "data_dir": "../../data/Raw/Chip2-NaCl 0-19_",           # str, Image directory
    "output_path": "./results.csv",                          # str, Path to output .csv file to track results across experiments (will be created if it doesnt already exist)
    "image_batch": f"290-4-0.75x-{chip_region}",             # str, Specific image batch, e.g. 290-4-0.75x-3
    "zoom": 0.75,                                            # float, Camera zoom
    "cc": True,                                              # bool, Use camera calibration
    "ffc": True,                                             # bool, Use flatfield correction
    "model": "SAM2",                                         # str, "SAM2" or "CellposeSAM"
    "sam2_config": model_cfg_b_plus,                         # str, path to specific SAM2 config (usually not necessary to change)
    "sam2_checkpoint": finetuned_checkpoint,                 # str, path to specific SAM2 checkpoint
    "approx": True,                                          # bool, Use postprocessed masks
    "approx_method": "circle",                              # str, use "circle" or "ellipse" for postprocessing
    "shrinking_factor": {                                    # dict[str, int], scaling factor applied to approximated masks. (For Chip2 data these are the optimal settings.)
        "large": 0.9,
        "medium": 0.8, 
        "small": 0.6, 
    },
    "overlay_masks": False,                                                # bool, use masks from first image on all following images.
    "individual_intensities": False,                                       # bool, show individual intensity points on intensity plots and link to segmentation results.
    "filter_config": {                                                     # dict, contains various configuration parameters for the segmentation mask filter.
        "circularity_threshold": 0.7,                                      # int, the circularity threshold used to filter irregular masks.
        "border_thres": 2,                                                 # int, the min number of pixels a mask has to be away from the border.
        "max_groups": 3 if chip_region == 1 or chip_region == 3 else 2,    # int (2 or 3), the number of well sizes. If None, the filter will determine the number of clusters (not recommended) .
        "eps_area": 20,                                                    # float, the epsilon parameter for the DBSCAN area filter applied to small wells. For Chip2 data used here this value performs well, but can always be tuned.
        "eps_coords_medium": 90,                                           # float, the epsilon parameter for the DBSCAN distance filter applied to medium wells. 
        "eps_coords_small": 70,                                            # float, the epsilon parameter for the DBSCAN distance filter applied to small wells.
        "eps_coords_small_exclude": 20,                                    # float, the epsilon parameter for the DBSCAN distance filter applied to remove tiny clusters (outliers are kept).
        "rough_only": False,                                               # bool, apply rough filters only
        "debug": True                                                     # bool, output additional visual debug information of individual steps
    },
    "calibrator_config": {                                                 # dict, contains various configuration parameters for the camera calibrator (perspective correction).
        "ref_radii": {                                                     # dict[str, int], contains the reference radii to be used for step 1 of the perspective correction (affine transformation).
            "region1": 500, # use large wells
            "region2": 190, # use medium wells
            "region3": 500 # use large wells
        },
        "schematic_path1": "../../data/Raw/Chip-schematic/chip_cleaned_1_10.jpg",        # str, path to schematic to be used for chip region 1
        "schematic_path2": "../../data/Raw/Chip-schematic/chip_cleaned_2_8.jpg",         # str, path to schematic to be used for chip region 2
        "schematic_path3": "../../data/Raw/Chip-schematic/chip_cleaned_3_10.jpg",        # str, path to schematic to be used for chip region 3
        "calculate_individual": False,                                                   # bool, if true the perspective transformation is calculated for each individual image. If false, it is calculated once and applied to following images.
        "chip_region": chip_region,                                                      
        "debug": False                                                     # bool, output additional visual debug information of individual steps
    },
    "debug": True                                                          # bool, output additional visual debug information
}

## Data Loading and Object Initialization

In [ ]:
from analysis.data_preprocessing import data_preprocessing
from analysis.camera_calibration import *
import json
from segmentation.segmenter import *

camera = Camera(
    zoom=EXPERIMENT_CONFIG["zoom"],
    um_per_pixel=6.45
)

# Collect the right images from the specified directory and group them in batches of increasing NaCl concentration
data = data_preprocessing(EXPERIMENT_CONFIG["data_dir"], EXPERIMENT_CONFIG["image_batch"])

print(json.dumps(data, indent=4))

# Isolate the specified batch as a list of dictionaries
data = data[EXPERIMENT_CONFIG["image_batch"] + ".jpg"]

# Specify model checkpoint and config (finetuned checkpoints use the b+ config)
sam2 = SAM2Wrapper(config=EXPERIMENT_CONFIG["sam2_config"], checkpoint=EXPERIMENT_CONFIG["sam2_checkpoint"], points_per_side=128)
cellposeSAM = CellposeSAMWrapper()

if EXPERIMENT_CONFIG["model"] == "SAM2":
    model = sam2
elif EXPERIMENT_CONFIG["model"] == "CellposeSAM":
    model = cellposeSAM
else:
    raise ValueError("Invalid model passed in EXPERIMENT_CONFIG")

model = Segmenter(model=model, filter_config=EXPERIMENT_CONFIG["filter_config"])

## Preprocessing: Flatfield Correction and Perspective Correction

In [ ]:
import os
import cv2
from PIL import Image
import numpy as np
from tqdm.notebook import tqdm
from segmentation.segmenter import *
from analysis.utils import apply_preprocessing

# Initialize Camera Calibrator for Perspective correction
setup_image_path = data[0]["file_path"]
setup_image = np.array(Image.open(setup_image_path).convert('L'))

schematic_path1 = EXPERIMENT_CONFIG["calibrator_config"]["schematic_path1"] 
schematic_raw1 = np.array(Image.open(schematic_path1).convert('L'))
schematic_mask1 = extract_schematic(image=schematic_raw1, debug=EXPERIMENT_CONFIG["debug"])
schematic_path2 = EXPERIMENT_CONFIG["calibrator_config"]["schematic_path2"] 
schematic_raw2 = np.array(Image.open(schematic_path2).convert('L'))
schematic_mask2 = extract_schematic(image=schematic_raw2, debug=EXPERIMENT_CONFIG["debug"])
schematic_path3 = EXPERIMENT_CONFIG["calibrator_config"]["schematic_path3"] 
schematic_raw3 = np.array(Image.open(schematic_path3).convert('L'))
schematic_mask3 = extract_schematic(image=schematic_raw3, debug=EXPERIMENT_CONFIG["debug"])
ref_images = {
    "region1": schematic_mask1,
    "region2": schematic_mask2,
    "region3": schematic_mask3,
}

calibrator = CameraCalibrator(
    setup_image=setup_image,
    camera=camera,
    segmenter=model,
    ref_images=ref_images,
    ref_radii=EXPERIMENT_CONFIG["calibrator_config"]["ref_radii"],
    calculate_individual=EXPERIMENT_CONFIG["calibrator_config"]["calculate_individual"]
)


# Apply preprocessing to all images in the batch
preprocessed_data = []
for image_data in tqdm(data):
    # Open Image
    gray_image = np.array(Image.open(image_data["file_path"]).convert("L"))

    # Noise Correction and Signal Enhancement
    preprocessed = apply_preprocessing(
        image=gray_image, 
        calibrator=calibrator,
        cc=EXPERIMENT_CONFIG["cc"], 
        ffc=EXPERIMENT_CONFIG["ffc"], 
        chip_region=EXPERIMENT_CONFIG["calibrator_config"]["chip_region"],
        debug=EXPERIMENT_CONFIG["calibrator_config"]["debug"]
    )

    # Store Results for the next step
    preprocessed_data.append({
        "image_original": gray_image,
        "image_enhanced": preprocessed,
        "file_path": image_data["file_path"],
        "nacl_percentage": image_data["nacl_percentage"]
    })

## Image Segmentation

In [ ]:
from segmentation.mask_container import MaskContainer
segmented = False

segmentation_data = []
for item in tqdm(preprocessed_data):
    image_enhanced = item["image_enhanced"]
    print(f"==== NaCl concentration: {item['nacl_percentage']} ====")

    if EXPERIMENT_CONFIG["overlay_masks"]:
        if not segmented:
            model.generate_masks(image_enhanced)
            model.visualize_masks()
            model.filter_masks()
            model.visualize_masks_by_well_size()
            large_mc, medium_mc, small_mc = model.large_masks, model.medium_masks, model.small_masks
            segmented = True
        model.set_image(image_enhanced)
        model.large_masks, model.medium_masks, model.small_masks = large_mc, medium_mc, small_mc
    else:
        model.generate_masks(image_enhanced)
        model.visualize_masks()
        model.filter_masks()

    model.visualize_masks_by_well_size()
    
    segmentation_data.append({
        "image": item["image_original"],
        "image_enhanced": image_enhanced,
        "nacl_percentage": item["nacl_percentage"],
        "masks_large": model.large_masks,
        "masks_medium": model.medium_masks,
        "masks_small": model.small_masks
    })

## Mask Postprocessing

In [ ]:
for item in tqdm(segmentation_data):
    item["masks_large"].use_approximated(EXPERIMENT_CONFIG["approx"], EXPERIMENT_CONFIG["approx_method"], shrink_factor=EXPERIMENT_CONFIG["shrinking_factor"]["large"])
    item["masks_medium"].use_approximated(EXPERIMENT_CONFIG["approx"], EXPERIMENT_CONFIG["approx_method"], shrink_factor=EXPERIMENT_CONFIG["shrinking_factor"]["medium"])
    item["masks_small"].use_approximated(EXPERIMENT_CONFIG["approx"], EXPERIMENT_CONFIG["approx_method"], shrink_factor=EXPERIMENT_CONFIG["shrinking_factor"]["small"])
            

postprocessed_data = segmentation_data    

## Intensity Calculation

In [ ]:
from analysis.analysis import intensity_analysis


intensity_data = []

for item in tqdm(postprocessed_data):
    intensity_data_large = intensity_analysis(item["image_enhanced"], item["masks_large"])
    intensity_data_medium = intensity_analysis(item["image_enhanced"], item["masks_medium"])
    intensity_data_small = intensity_analysis(item["image_enhanced"], item["masks_small"])

    intensity_data.append({
        "image": item["image"],
        "image_enhanced": item["image_enhanced"],
        "nacl_percentage": item["nacl_percentage"],
        "large": intensity_data_large,
        "medium": intensity_data_medium,
        "small": intensity_data_small
    })

## Sensitivity Analysis Results

In [ ]:
from analysis import utils
import pandas as pd

calibration_data = []
calibration_containers = {
    "large": [],
    "medium": [],
    "small": []
}
for item in tqdm(intensity_data):
    row = {
        'NaCl_percentage': item["nacl_percentage"],
        'image': item["image_enhanced"],
        'mean_intensity_large': item["large"]["mean_intensity"],
        'std_intensity_large': item["large"]["std_intensity"],
        'mean_intensity_medium': item["medium"]["mean_intensity"],
        'std_intensity_medium': item["medium"]["std_intensity"],
        'mean_intensity_small': item["small"]["mean_intensity"],
        'std_intensity_small': item["small"]["std_intensity"],
        'mean_intensity_all': (item["large"]["mean_intensity"] + item["medium"]["mean_intensity"] + item["small"]["mean_intensity"]) / 3,
        'std_intensity_all': (item["large"]["std_intensity"] + item["medium"]["std_intensity"] + item["small"]["std_intensity"]) / 3,
        'masks_large': item["large"]["masks"],
        'masks_medium': item["medium"]["masks"],
        'masks_small': item["small"]["masks"],
    }
    calibration_containers["large"].append(item["large"]["masks"])
    calibration_containers["medium"].append(item["medium"]["masks"])
    calibration_containers["small"].append(item["small"]["masks"])
    calibration_data.append(row)

df_data = pd.DataFrame(calibration_data).sort_values('NaCl_percentage')
# Fill NaN values with 0 if the entire column is NaN
df_data.loc[:, df_data.isna().all()] = 0 
# Compute refractive index (RI) using a simplified linear model
df_data['RI'] = 1.3330 + 0.0018 * df_data['NaCl_percentage']
# Drop rows with NaN values
df_calibration = df_data.dropna()

### Mask Visualization ###
utils.create_mask_plots(calibration_data, identify=EXPERIMENT_CONFIG["individual_intensities"])

avg_std_all = df_calibration["std_intensity_all"].astype(float).mean()
avg_std_large = df_calibration["std_intensity_large"].astype(float).mean()
avg_std_medium = df_calibration["std_intensity_medium"].astype(float).mean()
avg_std_small = df_calibration["std_intensity_small"].astype(float).mean()

print(f"==== Average std all: {avg_std_all} ==== ")
print(f"==== Average std large: {avg_std_large} ==== ")
print(f"==== Average std medium: {avg_std_medium} ==== ")
print(f"==== Average std small: {avg_std_small} ==== ")

settings = ""
if EXPERIMENT_CONFIG["ffc"]:
    settings += "ffc"
    
if EXPERIMENT_CONFIG["cc"]:
    settings += "cc"

if EXPERIMENT_CONFIG["approx"]:
    settings += "approx"
    
if settings == "":
    settings = "raw"

### Intensity vs RI Plots ###
slopes = utils.create_intensity_plots(
    df_calibration, 
    well_sizes=["large", "medium", "small"], 
    mask_containers=calibration_containers,
    individual_intensities=EXPERIMENT_CONFIG["individual_intensities"],
    settings=settings,
    csv_path=EXPERIMENT_CONFIG["output_path"],
    model=EXPERIMENT_CONFIG["model"]
)